In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('datathon-2026-round-1')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/datathon-2026-round-1


In [2]:
import pandas as pd

In [3]:
# Master
products_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/products.csv")
customers_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/customers.csv")
promotions_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/promotions.csv")
geography_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/geography.csv")

# Transaction
orders_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/orders.csv")
order_items_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/order_items.csv", low_memory=False)
payments_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/payments.csv")
shipments_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/shipments.csv")
returns_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/returns.csv")
reviews_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/reviews.csv")

# Operational
inventory_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/inventory.csv")
web_traffic_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/web_traffic.csv")

# Analytical
sales_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/sales.csv")
sample_submission_df = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/sample_submission.csv")

In [4]:
# Question 1
orders_tmp = orders_df.sort_values(["customer_id", "order_date"]).copy()
orders_tmp["order_date"] = pd.to_datetime(orders_tmp["order_date"])
orders_tmp["prev_order_date"] = orders_tmp.groupby("customer_id")["order_date"].shift(1)
orders_tmp["gap_days"] = (orders_tmp["order_date"] - orders_tmp["prev_order_date"]).dt.days
inter_order_gap = orders_tmp["gap_days"].dropna().median()

# Question 2
products_tmp = products_df.copy()
products_tmp["gross_margin"] = (products_tmp["price"] - products_tmp["cogs"]) / products_tmp["price"]
segment_margin = products_tmp.groupby("segment")["gross_margin"].mean()
best_segment = segment_margin.idxmax()
best_segment_value = segment_margin.max()

# Question 3
merged = returns_df.merge(products_df, on="product_id", how="inner")
streetwear_returns = merged[merged["category"] == "Streetwear"]
reason_count = streetwear_returns["return_reason"].value_counts()
top_reason = reason_count.idxmax()
top_count = reason_count.max()

# Question 4
source_bounce = web_traffic_df.groupby("traffic_source")["bounce_rate"].mean()
best_source = source_bounce.idxmin()
best_source_value = source_bounce.min()

# Question 5
total_rows = len(order_items_df)
promo_rows = order_items_df["promo_id"].notna().sum()
percent = promo_rows / total_rows * 100

# Question 6
age_group = customers_df[customers_df["age_group"].notna()]
merged = age_group.merge(orders_df, on="customer_id", how="left")
g = merged.groupby("age_group")
group = g["order_id"].count() / g["customer_id"].nunique()
best_group =group.idxmax()
best_group_value = group.max()

# Question 7
merged = order_items_df.merge(orders_df[["order_id", "zip"]], on="order_id", how="left")
merged = merged.merge(geography_df[["zip", "region"]], on="zip", how="left")
merged["line_revenue"] = merged["quantity"] * merged["unit_price"]
region_revenue = merged.groupby("region")["line_revenue"].sum()
best_region = region_revenue.idxmax()
best_region_value = region_revenue.max()

# Question 8
cancelled_orders = orders_df[orders_df['order_status'] == 'cancelled']
payments = cancelled_orders['payment_method'].value_counts()
most_payment = payments.idxmax()
most_value = payments.max()

# Question 9
items = order_items_df.merge(products_df, on='product_id', how='left')
returns = returns_df.merge(products_df, on='product_id', how='left')
returned = returns['size'].value_counts()
total = items['size'].value_counts()
rate = returned / total
highest_size = rate.idxmax()
highest_rate = rate.max()

# Question 10
install_plans = payments_df.groupby('installments')['payment_value'].mean()
best_plan = install_plans.idxmax()
best_plan_value = install_plans.max()

# Conclusion
print(" Q1: C", f"({inter_order_gap})")
print(" Q2: D", f"({best_segment} - {best_segment_value})")
print(" Q3: B", f"({top_reason} - {top_count})")
print(" Q4: C", f"({best_source} - {best_source_value})")
print(" Q5: C", f"{percent:.2f}%")
print(" Q6: A", f"({best_group} - {best_group_value})")
print(" Q7: C", f"({best_region} - {best_region_value})")
print(" Q8: A", f"({most_payment} - {most_value})")
print(" Q9: A", f"({highest_size} - {highest_rate})")
print("Q10: C", f"({best_plan} - {best_plan_value})")

 Q1: C (144.0)
 Q2: D (Standard - 0.31344174843884803)
 Q3: B (wrong_size - 7626)
 Q4: C (email_campaign - 0.0044584356435643565)
 Q5: C 38.66%
 Q6: A (55+ - 5.406851452775507)
 Q7: C (East - 7637532676.2)
 Q8: A (credit_card - 28452)
 Q9: A (S - 0.05651526952720847)
Q10: C (6 - 24446.65440296606)
